In [57]:
#%pip install pettingzoo

In [58]:
import os
import xml.etree.ElementTree as ET
import gymnasium.envs.mujoco as mujoco_envs

BASE_PATH  = os.path.dirname(mujoco_envs.__file__)
ASSETS_DIR = os.path.join(BASE_PATH, "assets")

def parse_agent(xml_path, prefix, spawn_pos):
    """
    Extrae bodies, actuadores, assets, defaults, contactos y tendones
    de un XML de MuJoCo, añadiendo un prefijo a todos los nombres
    para evitar colisiones entre agentes.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    NAME_ATTRS = ["name", "joint", "geom", "body", "mesh", "texture",
                  "material", "site", "tendon", "class", "childclass",
                  "actuator", "target", "cranksite", "slidersite"]

    def prefixify(elem):
        for attr in NAME_ATTRS:
            if attr in elem.attrib:
                val = elem.attrib[attr]
                # Evitamos doble prefijo
                if not val.startswith(prefix + "/"):
                    elem.attrib[attr] = f"{prefix}/{val}"
        for child in elem:
            prefixify(child)

    prefixify(root)

    # ── Worldbody: primer <body> hijo, reposicionado al carril
    bodies = []
    wb = root.find("worldbody")
    if wb is not None:
        for body in list(wb):
            # Eliminamos planos de suelo propios del agente
            for geom in list(body.findall(".//geom")):
                if geom.get("type") == "plane":
                    parent = body.find(f".//*[@name='{geom.get('name')}']/..")
                    # Búsqueda robusta del padre
                    for p in body.iter():
                        if geom in list(p):
                            p.remove(geom)
                            break
            body.attrib["pos"] = spawn_pos
            bodies.append(body)

    # ── Actuadores
    actuators = []
    act_sec = root.find("actuator")
    if act_sec is not None:
        actuators = list(act_sec)

    # ── Assets: resolvemos rutas relativas a absolutas
    assets = []
    asset_sec = root.find("asset")
    if asset_sec is not None:
        for elem in list(asset_sec):
            if "file" in elem.attrib:
                fname = elem.attrib["file"]
                if not os.path.isabs(fname):
                    full = os.path.join(ASSETS_DIR, fname)
                    if os.path.exists(full):
                        elem.attrib["file"] = full
            assets.append(elem)

    # ── Default: lo envolvemos en <default class="prefix">
    # para que no colisionen motores/geoms entre agentes
    default_wrapper = None
    def_sec = root.find("default")
    if def_sec is not None:
        default_wrapper = ET.Element("default", attrib={"class": prefix})
        for child in list(def_sec):
            default_wrapper.append(child)

    # ── Contactos
    contacts = []
    c_sec = root.find("contact")
    if c_sec is not None:
        contacts = list(c_sec)

    # ── Igualdades
    equalities = []
    eq_sec = root.find("equality")
    if eq_sec is not None:
        equalities = list(eq_sec)

    # ── Tendones
    tendons = []
    t_sec = root.find("tendon")
    if t_sec is not None:
        tendons = list(t_sec)

    return bodies, actuators, assets, default_wrapper, contacts, equalities, tendons


# ── Parseamos los tres agentes
agents_cfg = [
    ("ant",      os.path.join(ASSETS_DIR, "ant.xml"),           "0  4    0.75"),
    ("cheetah",  os.path.join(ASSETS_DIR, "half_cheetah.xml"),  "0  0    0.5"),
    ("humanoid", os.path.join(ASSETS_DIR, "humanoid.xml"),      "0 -4    1.25"),
]

parsed = {}
for name, path, pos in agents_cfg:
    parsed[name] = parse_agent(path, name, pos)

# ── Construimos el XML maestro
root_elem = ET.Element("mujoco", attrib={"model": "ai_olympics"})

ET.SubElement(root_elem, "compiler", attrib={
    "angle": "degree",
    "inertiafromgeom": "true",
    "coordinate": "local",
})
ET.SubElement(root_elem, "option", attrib={
    "timestep": "0.01",
    "integrator": "RK4",
    "gravity": "0 0 -9.81",
})

# Assets
asset_elem = ET.SubElement(root_elem, "asset")
for name, _, __ in agents_cfg:
    for a in parsed[name][2]:
        asset_elem.append(a)

# Defaults — cada agente en su propia <default class="agente">
default_root = ET.SubElement(root_elem, "default")
for name, _, __ in agents_cfg:
    wrapper = parsed[name][3]
    if wrapper is not None:
        default_root.append(wrapper)

# Worldbody
wb_elem = ET.SubElement(root_elem, "worldbody")
ET.SubElement(wb_elem, "light", attrib={
    "directional": "true", "diffuse": ".8 .8 .8",
    "specular": ".2 .2 .2", "pos": "0 0 10", "dir": "0 0 -1",
})
ET.SubElement(wb_elem, "geom", attrib={
    "name": "main_floor", "type": "plane",
    "pos": "50 0 0", "size": "60 12 0.1",
    "rgba": "0.15 0.25 0.35 1", "condim": "3",
})
# Líneas de carril decorativas
for y, rgba in [("4", "1 0.85 0 0.5"), ("0", "0 0.85 1 0.5"), ("-4", "1 0.4 0.4 0.5")]:
    ET.SubElement(wb_elem, "geom", attrib={
        "type": "box", "pos": f"50 {y} 0.015",
        "size": "55 0.06 0.005", "rgba": rgba, "contype": "0", "conaffinity": "0",
    })

for name, _, __ in agents_cfg:
    for body in parsed[name][0]:
        wb_elem.append(body)

# Contactos
contact_elem = ET.SubElement(root_elem, "contact")
for name, _, __ in agents_cfg:
    for c in parsed[name][4]:
        contact_elem.append(c)

# Igualdades
eq_elem = ET.SubElement(root_elem, "equality")
for name, _, __ in agents_cfg:
    for e in parsed[name][5]:
        eq_elem.append(e)

# Tendones
tendon_elem = ET.SubElement(root_elem, "tendon")
for name, _, __ in agents_cfg:
    for t in parsed[name][6]:
        tendon_elem.append(t)

# Actuadores
act_elem = ET.SubElement(root_elem, "actuator")
for name, _, __ in agents_cfg:
    for a in parsed[name][1]:
        act_elem.append(a)

# ── Guardar
tree = ET.ElementTree(root_elem)
ET.indent(tree, space="  ")
tree.write("arena.xml", encoding="unicode", xml_declaration=False)

print("✅ arena.xml generado.")

# Verificación rápida
import mujoco
try:
    m = mujoco.MjModel.from_xml_path("arena.xml")
    print(f"✅ Modelo OK — nq={m.nq}  nv={m.nv}  nu={m.nu}  nbody={m.nbody}")
except Exception as e:
    print(f"❌ {e}")
    # Muestra el XML para depurar
    with open("arena.xml") as f:
        print(f.read()[:4000])

✅ arena.xml generado.
✅ Modelo OK — nq=48  nv=46  nu=31  nbody=34


In [59]:
import mujoco

try:
    model = mujoco.MjModel.from_xml_path("arena.xml")
    print(f"✅ Modelo cargado:")
    print(f"   - nq (posiciones generalizadas): {model.nq}")
    print(f"   - nv (velocidades):              {model.nv}")
    print(f"   - nu (actuadores):               {model.nu}")
    print(f"   - nbody:                         {model.nbody}")
except Exception as e:
    print(f"❌ Error: {e}")
    # Si falla, imprime el XML para inspección
    with open("arena.xml") as f:
        print(f.read()[:3000])

✅ Modelo cargado:
   - nq (posiciones generalizadas): 48
   - nv (velocidades):              46
   - nu (actuadores):               31
   - nbody:                         34


In [60]:
import functools
import mujoco
import numpy as np
import gymnasium as gym
from pettingzoo import ParallelEnv

class OlympicEnv(ParallelEnv):
    metadata = {"render_modes": ["human", "rgb_array"], "name": "olympic_v0"}

    def __init__(self, render_mode=None):
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path("arena.xml")
        self.data  = mujoco.MjData(self.model)

        self.possible_agents = ["ant", "cheetah", "humanoid"]
        self.agents = self.possible_agents[:]

        # ── Índices de actuadores en self.data.ctrl
        # Verificamos con model.nu para no hardcodear a ciegas
        nu = self.model.nu
        print(f"Total actuadores en el modelo: {nu}")

        # Ant: 8 actuadores, Cheetah: 6, Humanoid: el resto
        self._ctrl_slices = {
            "ant":      slice(0, 8),
            "cheetah":  slice(8, 14),
            "humanoid": slice(14, nu),   # flexible
        }
        self._act_sizes = {
            "ant":      8,
            "cheetah":  6,
            "humanoid": nu - 14,
        }

        # ── Índices de qpos para calcular avance en X
        # ant: qpos[0..6] = free joint (x,y,z,quat), cheetah: qpos[7..13], etc.
        # Usamos mujoco para buscar por nombre de joint
        self._x_qpos_idx = {}
        for agent, joint_name in [("ant", "ant/root"), ("cheetah", "cheetah/rootx"),
                                   ("humanoid", "humanoid/root")]:
            try:
                jid = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
                self._x_qpos_idx[agent] = self.model.jnt_qposadr[jid]
            except:
                self._x_qpos_idx[agent] = 0  # fallback

        print("Índices X en qpos:", self._x_qpos_idx)

    @functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        obs_sizes = {
            "ant":      27,
            "cheetah":  17,
            "humanoid": 376,
        }
        return gym.spaces.Box(-np.inf, np.inf,
                              shape=(obs_sizes[agent],), dtype=np.float64)

    @functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        return gym.spaces.Box(-1.0, 1.0,
                              shape=(self._act_sizes[agent],), dtype=np.float32)

    def _get_obs(self, agent):
        """Concatena qpos + qvel para el agente (simplificado)."""
        size = self.observation_space(agent).shape[0]
        # Rellena con lo disponible y completa con ceros si es necesario
        full = np.concatenate([self.data.qpos, self.data.qvel])
        obs = full[:size] if len(full) >= size else np.pad(full, (0, size - len(full)))
        return obs.astype(np.float64)

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        mujoco.mj_resetData(self.model, self.data)
        self.agents = self.possible_agents[:]
        self._prev_x = {agent: self.data.qpos[self._x_qpos_idx[agent]]
                        for agent in self.agents}
        observations = {agent: self._get_obs(agent) for agent in self.agents}
        return observations, {}

    def step(self, actions):
        # Aplicar acciones
        for agent, action in actions.items():
            s = self._ctrl_slices[agent]
            clipped = np.clip(action, -1.0, 1.0)
            self.data.ctrl[s] = clipped

        mujoco.mj_step(self.model, self.data)

        # Recompensa = avance en X desde el paso anterior
        rewards = {}
        for agent in self.agents:
            x_now  = self.data.qpos[self._x_qpos_idx[agent]]
            x_prev = self._prev_x.get(agent, 0.0)
            rewards[agent] = float(x_now - x_prev)
            self._prev_x[agent] = x_now

        observations  = {agent: self._get_obs(agent) for agent in self.agents}
        terminations  = {agent: False for agent in self.agents}
        truncations   = {agent: False for agent in self.agents}
        infos         = {agent: {} for agent in self.agents}

        return observations, rewards, terminations, truncations, infos

print("✅ OlympicEnv definida.")

✅ OlympicEnv definida.


In [61]:
import mujoco.viewer
import time

env = OlympicEnv()
observations, _ = env.reset()

with mujoco.viewer.launch_passive(env.model, env.data) as viewer:
    start = time.time()
    step  = 0

    while viewer.is_running() and (time.time() - start) < 30:
        actions = {
            "ant":      env.action_space("ant").sample(),
            "cheetah":  env.action_space("cheetah").sample(),
            "humanoid": env.action_space("humanoid").sample(),
        }
        observations, rewards, terminations, truncations, infos = env.step(actions)

        if step % 100 == 0:
            print(f"t={step:4d} | recompensas: " +
                  " | ".join(f"{a}: {r:+.3f}" for a, r in rewards.items()))

        viewer.sync()
        time.sleep(0.01)
        step += 1

print("🏁 Carrera finalizada.")

Total actuadores en el modelo: 31
Índices X en qpos: {'ant': np.int32(0), 'cheetah': np.int32(15), 'humanoid': np.int32(24)}
t=   0 | recompensas: ant: +0.000 | cheetah: +0.005 | humanoid: +0.006
t= 100 | recompensas: ant: -0.003 | cheetah: +0.003 | humanoid: +0.095
t= 200 | recompensas: ant: -0.001 | cheetah: -0.003 | humanoid: +0.009
t= 300 | recompensas: ant: +0.001 | cheetah: -0.000 | humanoid: -0.012
t= 400 | recompensas: ant: -0.001 | cheetah: +0.001 | humanoid: -0.007
t= 500 | recompensas: ant: +0.001 | cheetah: +0.001 | humanoid: +0.001
t= 600 | recompensas: ant: -0.001 | cheetah: -0.005 | humanoid: +0.011
t= 700 | recompensas: ant: +0.000 | cheetah: -0.000 | humanoid: -0.004
t= 800 | recompensas: ant: +0.002 | cheetah: +0.001 | humanoid: +0.004
t= 900 | recompensas: ant: +0.000 | cheetah: +0.001 | humanoid: -0.005
t=1000 | recompensas: ant: +0.002 | cheetah: +0.001 | humanoid: -0.023
t=1100 | recompensas: ant: -0.000 | cheetah: +0.002 | humanoid: -0.039
t=1200 | recompensas: a